In [1]:
import os
import re
import random
from tqdm import tqdm

random.seed(42)
CHARACTER_MAP = "gkamztlbdqiyfucxbhsjoprnweygtjmevchdxsanqolkrvwiypjzquhe"

def encode_line(text: str) -> str:
    return "".join(CHARACTER_MAP[ord(c) % 56] if c not in "\n\t\r" else c for c in text)

RAW_WIKI = "corpus/raw_wiki.txt"
RAW_EF = "corpus/raw_ef.txt"
OUT_DIR = "corpus"
os.makedirs(OUT_DIR, exist_ok=True)

## Load & Normalize Raw Lines

In [2]:
def process_and_save(raw_path, zh_out, skz_out):
    if not os.path.exists(raw_path):
        print(f"Warning: {raw_path} not found. Skipping.")
        return []
        
    with open(raw_path, "r", encoding="utf-8") as f:
        raw_lines = [line.strip() for line in f if line.strip()]

    valid_zh = []
    valid_skz = []
    for line in raw_lines:
        clean = re.sub(r"\s+", " ", line)
        if 20 <= len(clean) <= 200:
            if not re.search(r"[。！？；]$", clean):
                clean += "。"
            valid_zh.append(clean)
            valid_skz.append(encode_line(clean))

    with open(zh_out, "w", encoding="utf-8") as f_zh, \
         open(skz_out, "w", encoding="utf-8") as f_skz:
        for zh, skz in zip(valid_zh, valid_skz):
            f_zh.write(zh + "\n")
            f_skz.write(skz + "\n")

    print(f"Processed {os.path.basename(raw_path)}: {len(valid_zh)} pairs -> {zh_out}")
    return valid_zh

## Generate pure Wikipedia corpus (unmixed)

In [3]:
print("Generating pure Wikipedia corpus...")
wiki_zh = process_and_save(
    RAW_WIKI, 
    os.path.join(OUT_DIR, "wiki.zh"), 
    os.path.join(OUT_DIR, "wiki.skz")
)

Generating pure Wikipedia corpus...
Processed raw_wiki.txt: 17879714 pairs -> corpus/wiki.zh


## Generate pure Endfield corpus (unmixed)

In [4]:
print("Generating pure Endfield corpus...")
ef_zh = process_and_save(
    RAW_EF, 
    os.path.join(OUT_DIR, "endfield.zh"), 
    os.path.join(OUT_DIR, "endfield.skz")
)

Generating pure Endfield corpus...
Processed raw_ef.txt: 13096 pairs -> corpus/endfield.zh


## Combine, shuffle, and split into mixed train/val sets

In [5]:
print("Combining, shuffling, and splitting into train/val sets...")
combined_zh = wiki_zh + ef_zh
random.shuffle(combined_zh)

split_idx = int(len(combined_zh) * 0.95)
train_zh = combined_zh[:split_idx]
val_zh = combined_zh[split_idx:]

def write_split(zh_lines, prefix):
    skz_path = os.path.join(OUT_DIR, f"{prefix}.skz")
    zh_path = os.path.join(OUT_DIR, f"{prefix}.zh")
    with open(zh_path, "w", encoding="utf-8") as f_zh, \
         open(skz_path, "w", encoding="utf-8") as f_skz:
        for line in tqdm(zh_lines, desc=f"Writing {prefix}"):
            f_zh.write(line + "\n")
            f_skz.write(encode_line(line) + "\n")

write_split(train_zh, "train")
write_split(val_zh, "val")
print("Mixed train/val sets generated.")

Combining, shuffling, and splitting into train/val sets...


Writing val: 100%|██████████| 894641/894641 [00:05<00:00, 174221.41it/s]


Mixed train/val sets generated.


## Strict Consistency Check

In [6]:
print("Verifying file consistency...")
for prefix in ["wiki", "endfield", "train", "val"]:
    zh_path = os.path.join(OUT_DIR, f"{prefix}.zh")
    skz_path = os.path.join(OUT_DIR, f"{prefix}.skz")
    
    if not os.path.exists(zh_path) or not os.path.exists(skz_path):
        print(f"  {prefix}: Files missing. Skipping verification.")
        continue
        
    zh_count = len(open(zh_path, encoding="utf-8").readlines())
    skz_count = len(open(skz_path, encoding="utf-8").readlines())
    
    assert zh_count == skz_count, f"Mismatch in {prefix}: zh={zh_count} vs skz={skz_count}"
    print(f"  {prefix}: {zh_count} pairs (verified)")

print("All alignments verified successfully.")

Verifying file consistency...
  wiki: 17879714 pairs (verified)
  endfield: 13096 pairs (verified)
  train: 16998169 pairs (verified)
  val: 894641 pairs (verified)
All alignments verified successfully.
